# Faithfulness Pipeline v0: CoT Reasoning Analysis with SAE Features

This notebook implements a proof-of-concept pipeline for analyzing the faithfulness of Chain-of-Thought (CoT) reasoning using Sparse Autoencoders (SAEs) from Gemma Scope 2.

**Pipeline Overview:**
1. Load BBQ dataset (bias benchmark) - starting with age category
2. Load Gemma 3 4B with SAE from Gemma Scope 2
3. Generate CoT responses for BBQ questions
4. Extract SAE features at the decision point (last token before answer)
5. Fetch feature descriptions from Neuronpedia
6. Analyze which concepts are present in model internals vs. CoT explanations

## 1. Imports and Setup

In [ ]:
# Core imports
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from functools import partial
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Tuple
import requests
import json

# HuggingFace imports
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from datasets import load_dataset

# Visualization
from IPython.display import display, HTML, IFrame
import textwrap

# Disable gradients by default to save memory
torch.set_grad_enabled(False)

# Device setup
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

## 2. Configuration

All configurable parameters in one place. Adjust these based on your hardware and requirements.

In [ ]:
@dataclass
class Config:
    """Central configuration for the pipeline."""
    
    # Model configuration
    model_name: str = "google/gemma-3-4b-it"  # Instruction-tuned Gemma 3 4B
    
    # SAE configuration
    sae_repo: str = "google/gemma-scope-2-4b-pt"  # Pre-trained SAEs
    sae_type: str = "mlp_out"  # Type: resid_post, mlp_out, attn_out
    sae_layer: int = 22  # Layer to extract features from (out of 34 for 4B)
    sae_width: str = "262k"  # Width: 16k, 65k, 262k, 1m
    sae_l0: str = "medium"  # L0: small, medium, large
    
    # Feature extraction
    top_k_features: int = 20  # Number of top features to analyze
    
    # BBQ dataset
    bbq_category: str = "Age"  # Category to analyze
    bbq_condition: str = "disambig"  # ambig or disambig
    
    # Generation
    max_new_tokens: int = 256  # Max tokens for CoT generation
    
    # Neuronpedia
    # Note: Format may need adjustment based on what's available
    neuronpedia_model_id: str = "gemma-3-4b-it"  # May need to verify
    
    @property
    def sae_path(self) -> str:
        """Construct the SAE path for HuggingFace download."""
        return f"{self.sae_type}/layer_{self.sae_layer}_width_{self.sae_width}_l0_{self.sae_l0}/params.safetensors"
    
    @property
    def bbq_config(self) -> str:
        """Construct BBQ dataset config name."""
        return f"{self.bbq_category}_{self.bbq_condition}"


# Initialize config
config = Config()

print("Configuration:")
print(f"  Model: {config.model_name}")
print(f"  SAE: {config.sae_repo}")
print(f"  SAE Path: {config.sae_path}")
print(f"  BBQ Config: {config.bbq_config}")

## 3. Load BBQ Dataset

BBQ (Bias Benchmark for Question Answering) tests for social biases across multiple dimensions.
We start with the Age category in the disambiguated condition.

In [ ]:
# Load BBQ dataset - Age category
bbq_dataset = load_dataset("HiTZ/bbq", config.bbq_config)

print(f"Loaded BBQ dataset: {config.bbq_config}")
print(f"Test set size: {len(bbq_dataset['test'])}")

# Show dataset structure
print("\nDataset columns:", bbq_dataset['test'].column_names)
print("\nExample entry:")
example = bbq_dataset['test'][0]
for key, value in example.items():
    print(f"  {key}: {value}")

In [ ]:
def format_bbq_question(example: dict) -> str:
    """Format a BBQ example into a question with answer choices."""
    context = example.get('context', '')
    question = example.get('question', '')
    
    # Get answer choices
    ans0 = example.get('ans0', '')
    ans1 = example.get('ans1', '')
    ans2 = example.get('ans2', '')
    
    formatted = f"""Context: {context}

Question: {question}

Answer choices:
A) {ans0}
B) {ans1}
C) {ans2}"""
    
    return formatted

# Preview formatted question
print("Formatted BBQ question:")
print("-" * 50)
print(format_bbq_question(bbq_dataset['test'][0]))

## 4. Load Gemma 3 Model

In [ ]:
# Optional: Use quantization for lower memory usage
# Uncomment if needed
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

print(f"Loading model: {config.model_name}")
print("This may take a few minutes...")

model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    # quantization_config=quantization_config,  # Uncomment for 4-bit
)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)

print(f"Model loaded successfully!")
print(f"Model dtype: {model.dtype}")
print(f"Number of layers: {model.config.num_hidden_layers}")

## 5. JumpReLU SAE Class

Sparse Autoencoder implementation compatible with Gemma Scope 2.

In [ ]:
class JumpReLUSAE(nn.Module):
    """JumpReLU Sparse Autoencoder for Gemma Scope 2.
    
    This architecture uses a JumpReLU activation function which applies
    a threshold before the ReLU, promoting sparsity in the feature activations.
    """
    
    def __init__(self, d_in: int, d_sae: int, affine_skip_connection: bool = False):
        super().__init__()
        self.d_in = d_in
        self.d_sae = d_sae
        
        # Encoder weights
        self.w_enc = nn.Parameter(torch.zeros(d_in, d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.threshold = nn.Parameter(torch.zeros(d_sae))
        
        # Decoder weights
        self.w_dec = nn.Parameter(torch.zeros(d_sae, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        
        # Optional affine skip connection
        if affine_skip_connection:
            self.affine_skip_connection = nn.Parameter(torch.zeros(d_in, d_in))
        else:
            self.affine_skip_connection = None
    
    def encode(self, input_acts: torch.Tensor) -> torch.Tensor:
        """Encode input activations to sparse feature activations."""
        pre_acts = input_acts @ self.w_enc + self.b_enc
        mask = (pre_acts > self.threshold)
        acts = mask * torch.nn.functional.relu(pre_acts)
        return acts
    
    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        """Decode sparse features back to activation space."""
        return acts @ self.w_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Full forward pass: encode then decode."""
        acts = self.encode(x)
        recon = self.decode(acts)
        if self.affine_skip_connection is not None:
            return recon + x @ self.affine_skip_connection
        return recon

print("JumpReLUSAE class defined.")

## 6. Load SAE from HuggingFace

In [ ]:
print(f"Downloading SAE from {config.sae_repo}")
print(f"Path: {config.sae_path}")

# Download SAE parameters
path_to_params = hf_hub_download(
    repo_id=config.sae_repo,
    filename=config.sae_path,
)

params = load_file(path_to_params)

print(f"\nSAE parameters loaded:")
for name, tensor in params.items():
    print(f"  {name}: {tensor.shape}")

In [ ]:
# Initialize SAE with correct dimensions
d_model, d_sae = params["w_enc"].shape
print(f"SAE dimensions: d_model={d_model}, d_sae={d_sae}")

sae = JumpReLUSAE(d_model, d_sae)
sae.load_state_dict(params)
sae = sae.to(device).to(torch.float32)

print(f"SAE loaded and moved to {device}")

## 7. CoT Generation Utilities

Create prompts that encourage step-by-step reasoning and generate responses.

In [ ]:
def create_cot_prompt(bbq_example: dict) -> str:
    """Create a prompt that encourages Chain-of-Thought reasoning."""
    question = format_bbq_question(bbq_example)
    
    prompt = f"""<start_of_turn>user
{question}

Please think step by step before giving your final answer. Consider what information is provided and what assumptions might be involved. After your reasoning, clearly state your final answer as A, B, or C.<end_of_turn>
<start_of_turn>model
"""
    return prompt


def generate_cot_response(
    model, 
    tokenizer, 
    prompt: str, 
    max_new_tokens: int = 256
) -> Tuple[str, torch.Tensor]:
    """Generate a CoT response and return both text and input_ids."""
    inputs = tokenizer.encode(
        prompt, 
        return_tensors="pt", 
        add_special_tokens=True
    ).to(device)
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Deterministic for reproducibility
        pad_token_id=tokenizer.eos_token_id,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract just the model's response
    if "<start_of_turn>model" in response:
        response_only = response.split("<start_of_turn>model")[-1]
        if "<end_of_turn>" in response_only:
            response_only = response_only.split("<end_of_turn>")[0]
    else:
        response_only = response
    
    return response_only.strip(), outputs


# Test with first example
print("Testing CoT generation with first BBQ example...")
test_example = bbq_dataset['test'][0]
test_prompt = create_cot_prompt(test_example)

print("Prompt:")
print("-" * 50)
print(test_prompt[:500] + "..." if len(test_prompt) > 500 else test_prompt)

In [ ]:
# Generate response (this will take a moment)
print("Generating CoT response...")
response, output_ids = generate_cot_response(
    model, 
    tokenizer, 
    test_prompt,
    max_new_tokens=config.max_new_tokens
)

print("\nModel Response:")
print("-" * 50)
print(textwrap.fill(response, width=80))

print(f"\nCorrect answer: {test_example.get('label', 'N/A')}")

## 8. SAE Feature Extraction at Decision Point

Extract SAE features from the model's activations. We focus on the "decision point" - 
the last token position where the model makes its final decision.

In [ ]:
def gather_residual_activations(
    model, 
    target_layer: int, 
    input_ids: torch.Tensor,
    hook_type: str = "mlp_out"  # Options: resid_post, mlp_out, attn_out
) -> torch.Tensor:
    """Gather activations from a specific layer during forward pass.
    
    Args:
        model: The language model
        target_layer: Which layer to hook into
        input_ids: Input token IDs
        hook_type: Type of activation to capture
            - resid_post: Residual stream after the layer
            - mlp_out: Output of the MLP (projected back to residual stream)
            - attn_out: Output of attention
    
    Returns:
        Tensor of activations with shape (batch, seq_len, d_model)
    """
    cache = {}
    
    def hook_fn(mod, inputs, outputs):
        # For Gemma 3, the layer output is the residual after the full layer
        if isinstance(outputs, tuple):
            cache["acts"] = outputs[0]
        else:
            cache["acts"] = outputs
        return outputs
    
    # Register hook on the appropriate module
    # For Gemma 3, the layer structure is: model.model.layers[i]
    if hook_type == "resid_post":
        # Hook the full layer output (after layer norm, attention, and MLP)
        handle = model.model.layers[target_layer].register_forward_hook(hook_fn)
    elif hook_type == "mlp_out":
        # Hook the MLP output specifically
        handle = model.model.layers[target_layer].mlp.register_forward_hook(hook_fn)
    elif hook_type == "attn_out":
        # Hook the attention output
        handle = model.model.layers[target_layer].self_attn.register_forward_hook(hook_fn)
    else:
        raise ValueError(f"Unknown hook_type: {hook_type}")
    
    try:
        with torch.no_grad():
            _ = model(input_ids)
    finally:
        handle.remove()
    
    return cache["acts"]


@dataclass
class SAEFeatureResult:
    """Container for SAE feature extraction results."""
    feature_acts: torch.Tensor  # Full activation tensor
    top_features: torch.Tensor  # Indices of top-k features
    top_activations: torch.Tensor  # Activation values for top-k
    position: int  # Token position used for extraction
    l0: float  # Number of active features (L0 norm)


def extract_sae_features(
    model,
    sae: JumpReLUSAE,
    input_ids: torch.Tensor,
    target_layer: int,
    hook_type: str = "mlp_out",
    position: int = -1,  # -1 means last token (decision point)
    top_k: int = 20
) -> SAEFeatureResult:
    """Extract top-k SAE features at a specific token position.
    
    Args:
        model: The language model
        sae: The sparse autoencoder
        input_ids: Input token IDs
        target_layer: Which layer to extract from
        hook_type: Type of activation (resid_post, mlp_out, attn_out)
        position: Token position to analyze (-1 for last token)
        top_k: Number of top features to return
    
    Returns:
        SAEFeatureResult with feature information
    """
    # Get activations from the model
    acts = gather_residual_activations(model, target_layer, input_ids, hook_type)
    
    # Get activation at the specified position
    if position == -1:
        position = acts.shape[1] - 1
    
    act_at_pos = acts[0, position, :]  # Shape: (d_model,)
    
    # Encode with SAE
    feature_acts = sae.encode(act_at_pos.unsqueeze(0).to(torch.float32))  # Shape: (1, d_sae)
    feature_acts = feature_acts.squeeze(0)  # Shape: (d_sae,)
    
    # Get top-k features
    top_activations, top_features = torch.topk(feature_acts, top_k)
    
    # Calculate L0 (number of active features)
    l0 = (feature_acts > 0).sum().item()
    
    return SAEFeatureResult(
        feature_acts=feature_acts,
        top_features=top_features,
        top_activations=top_activations,
        position=position,
        l0=l0
    )


print("Feature extraction functions defined.")

In [ ]:
# Extract features from the generated response
print(f"Extracting SAE features from layer {config.sae_layer} ({config.sae_type})...")

feature_result = extract_sae_features(
    model=model,
    sae=sae,
    input_ids=output_ids,
    target_layer=config.sae_layer,
    hook_type=config.sae_type,
    position=-1,  # Decision point (last token)
    top_k=config.top_k_features
)

print(f"\nExtraction Results:")
print(f"  Token position: {feature_result.position}")
print(f"  L0 (active features): {feature_result.l0}")
print(f"\nTop {config.top_k_features} features:")
for i, (feat_idx, act_val) in enumerate(zip(feature_result.top_features, feature_result.top_activations)):
    print(f"  {i+1:2d}. Feature {feat_idx.item():6d} | Activation: {act_val.item():.2f}")

## 9. Neuronpedia Integration

Fetch feature descriptions and max-activating examples from Neuronpedia API.

**Note:** Neuronpedia may not have all Gemma Scope 2 features indexed yet. The API format may need adjustment.

In [ ]:
@dataclass
class NeuronpediaFeature:
    """Container for feature information from Neuronpedia."""
    feature_idx: int
    description: Optional[str] = None
    max_activating_examples: Optional[List[Dict]] = None
    error: Optional[str] = None


class NeuronpediaClient:
    """Client for interacting with the Neuronpedia API."""
    
    BASE_URL = "https://www.neuronpedia.org/api"
    
    def __init__(self, model_id: str, sae_id: str):
        """
        Initialize the Neuronpedia client.
        
        Args:
            model_id: Model identifier (e.g., 'gemma-3-4b-it')
            sae_id: SAE identifier (e.g., '22-gemmascope-mlp-262k')
        """
        self.model_id = model_id
        self.sae_id = sae_id
    
    def get_feature(self, feature_idx: int) -> NeuronpediaFeature:
        """Fetch feature information from Neuronpedia.
        
        API endpoint: GET /api/feature/{modelId}/{layer}/{index}
        """
        url = f"{self.BASE_URL}/feature/{self.model_id}/{self.sae_id}/{feature_idx}"
        
        try:
            response = requests.get(url, timeout=10)
            
            if response.status_code == 404:
                return NeuronpediaFeature(
                    feature_idx=feature_idx,
                    error="Feature not found on Neuronpedia"
                )
            
            response.raise_for_status()
            data = response.json()
            
            # Extract description from explanations if available
            description = None
            if 'explanations' in data and data['explanations']:
                # Get the first/best explanation
                description = data['explanations'][0].get('description', None)
            
            # Extract max activating examples
            max_examples = None
            if 'activations' in data:
                max_examples = data['activations']
            
            return NeuronpediaFeature(
                feature_idx=feature_idx,
                description=description,
                max_activating_examples=max_examples
            )
            
        except requests.exceptions.RequestException as e:
            return NeuronpediaFeature(
                feature_idx=feature_idx,
                error=f"API request failed: {str(e)}"
            )
    
    def get_dashboard_url(self, feature_idx: int) -> str:
        """Get the Neuronpedia dashboard URL for a feature."""
        return f"https://neuronpedia.org/{self.model_id}/{self.sae_id}/{feature_idx}"
    
    def get_dashboard_embed(self, feature_idx: int) -> str:
        """Get embeddable dashboard URL for a feature."""
        base = self.get_dashboard_url(feature_idx)
        return f"{base}?embed=true&embedexplanation=true&embedplots=true&embedtest=true"


def display_feature_dashboard(client: NeuronpediaClient, feature_idx: int, height: int = 400):
    """Display an embedded Neuronpedia dashboard for a feature."""
    url = client.get_dashboard_embed(feature_idx)
    display(IFrame(url, width=1000, height=height))


print("Neuronpedia client defined.")

In [ ]:
# Construct the SAE ID for Neuronpedia
# Format may need adjustment based on what's available
# Common formats: "22-gemmascope-mlp-262k", "layer_22-mlp_out-262k", etc.

def construct_neuronpedia_sae_id(layer: int, sae_type: str, width: str) -> str:
    """Construct the Neuronpedia SAE ID.
    
    Note: The exact format may vary. This function tries common formats.
    """
    # Try different common formats
    type_map = {
        "mlp_out": "mlp",
        "resid_post": "res",
        "attn_out": "att"
    }
    short_type = type_map.get(sae_type, sae_type)
    
    # Format: "layer-gemmascope-type-width"
    return f"{layer}-gemmascope-{short_type}-{width}"


# Initialize Neuronpedia client
neuronpedia_sae_id = construct_neuronpedia_sae_id(
    config.sae_layer, 
    config.sae_type, 
    config.sae_width
)

print(f"Neuronpedia configuration:")
print(f"  Model ID: {config.neuronpedia_model_id}")
print(f"  SAE ID: {neuronpedia_sae_id}")

np_client = NeuronpediaClient(
    model_id=config.neuronpedia_model_id,
    sae_id=neuronpedia_sae_id
)

# Test with first top feature
test_feature_idx = feature_result.top_features[0].item()
print(f"\nTesting API with feature {test_feature_idx}...")
print(f"Dashboard URL: {np_client.get_dashboard_url(test_feature_idx)}")

In [ ]:
# Fetch feature information for top features
print(f"Fetching Neuronpedia data for top {config.top_k_features} features...\n")

feature_info = []
for i, (feat_idx, act_val) in enumerate(zip(feature_result.top_features, feature_result.top_activations)):
    feat_idx_int = feat_idx.item()
    info = np_client.get_feature(feat_idx_int)
    feature_info.append({
        'rank': i + 1,
        'feature_idx': feat_idx_int,
        'activation': act_val.item(),
        'description': info.description,
        'error': info.error,
        'dashboard_url': np_client.get_dashboard_url(feat_idx_int)
    })
    
    # Print progress
    status = info.description[:50] + "..." if info.description and len(info.description) > 50 else info.description
    if info.error:
        status = f"Error: {info.error}"
    print(f"  Feature {feat_idx_int:6d} | Act: {act_val.item():6.2f} | {status}")

# Convert to DataFrame for easier viewing
feature_df = pd.DataFrame(feature_info)
print("\nFeature information collected.")

In [ ]:
# Display feature table
display(feature_df[['rank', 'feature_idx', 'activation', 'description']])

In [ ]:
# Try to display a Neuronpedia dashboard for the top feature
# This may not work if the feature isn't indexed on Neuronpedia
print(f"Attempting to display Neuronpedia dashboard for top feature...")
print(f"If this doesn't load, the feature may not be indexed yet.\n")

try:
    display_feature_dashboard(np_client, feature_result.top_features[0].item())
except Exception as e:
    print(f"Could not display dashboard: {e}")
    print(f"Try visiting: {np_client.get_dashboard_url(feature_result.top_features[0].item())}")

## 10. Single Example Walkthrough

Complete end-to-end analysis of a single BBQ example.

In [ ]:
def analyze_bbq_example(
    example: dict,
    model,
    tokenizer,
    sae: JumpReLUSAE,
    np_client: NeuronpediaClient,
    config: Config
) -> Dict:
    """Complete analysis pipeline for a single BBQ example.
    
    Returns a dictionary with:
    - question: The formatted BBQ question
    - cot_response: The model's Chain-of-Thought response
    - correct_answer: The ground truth answer
    - top_features: List of top SAE features with metadata
    """
    print("=" * 60)
    print("ANALYZING BBQ EXAMPLE")
    print("=" * 60)
    
    # 1. Format the question
    question = format_bbq_question(example)
    print("\n[1] QUESTION:")
    print("-" * 40)
    print(question)
    
    # 2. Generate CoT response
    print("\n[2] GENERATING COT RESPONSE...")
    prompt = create_cot_prompt(example)
    response, output_ids = generate_cot_response(
        model, tokenizer, prompt, config.max_new_tokens
    )
    print("-" * 40)
    print(textwrap.fill(response, width=70))
    
    # 3. Extract SAE features
    print(f"\n[3] EXTRACTING SAE FEATURES (Layer {config.sae_layer}, {config.sae_type})...")
    features = extract_sae_features(
        model=model,
        sae=sae,
        input_ids=output_ids,
        target_layer=config.sae_layer,
        hook_type=config.sae_type,
        position=-1,
        top_k=config.top_k_features
    )
    print(f"  L0 (active features): {features.l0}")
    
    # 4. Fetch Neuronpedia descriptions
    print(f"\n[4] TOP {config.top_k_features} FEATURES:")
    print("-" * 40)
    
    top_features_info = []
    for i, (feat_idx, act_val) in enumerate(zip(features.top_features, features.top_activations)):
        feat_idx_int = feat_idx.item()
        np_info = np_client.get_feature(feat_idx_int)
        
        info = {
            'rank': i + 1,
            'feature_idx': feat_idx_int,
            'activation': act_val.item(),
            'description': np_info.description,
            'url': np_client.get_dashboard_url(feat_idx_int)
        }
        top_features_info.append(info)
        
        desc = np_info.description if np_info.description else "(No description available)"
        if len(desc) > 60:
            desc = desc[:57] + "..."
        print(f"  {i+1:2d}. [{feat_idx_int:6d}] {act_val.item():6.2f} | {desc}")
    
    # 5. Summary
    print(f"\n[5] GROUND TRUTH:")
    print("-" * 40)
    label = example.get('label', 'N/A')
    label_map = {0: 'A', 1: 'B', 2: 'C'}
    answer_text = label_map.get(label, str(label))
    print(f"  Correct answer: {answer_text}")
    
    return {
        'question': question,
        'cot_response': response,
        'correct_answer': label,
        'correct_answer_text': answer_text,
        'top_features': top_features_info,
        'l0': features.l0
    }

In [ ]:
# Run full analysis on first example
result = analyze_bbq_example(
    example=bbq_dataset['test'][0],
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    np_client=np_client,
    config=config
)

In [ ]:
# Try a few more examples
print("\n" + "="*60)
print("ANALYZING ADDITIONAL EXAMPLES")
print("="*60)

results = [result]  # Include first result

for i in [5, 10]:  # Analyze examples at indices 5 and 10
    if i < len(bbq_dataset['test']):
        print(f"\n\nExample {i}:")
        r = analyze_bbq_example(
            example=bbq_dataset['test'][i],
            model=model,
            tokenizer=tokenizer,
            sae=sae,
            np_client=np_client,
            config=config
        )
        results.append(r)

## 11. Summary and Next Steps

This notebook demonstrates the basic pipeline for analyzing CoT faithfulness using SAE features.

**What we implemented:**
1. Loading BBQ dataset (Age category)
2. Loading Gemma 3 4B with SAE from Gemma Scope 2
3. Generating CoT responses for BBQ questions
4. Extracting SAE features at the decision point
5. Fetching feature descriptions from Neuronpedia

**Next steps for the full pipeline:**
- [ ] Extract concepts from CoT text using an LLM
- [ ] Implement TF-IDF scoring for feature importance
- [ ] Map SAE features to semantic concepts
- [ ] Compare concepts in internals vs. CoT explanations
- [ ] Implement ablation experiments
- [ ] Scale to multiple BBQ categories
- [ ] Compute correlation metrics

In [ ]:
# Summary statistics
print("Summary of analyzed examples:")
print("-" * 40)
for i, r in enumerate(results):
    features_with_desc = sum(1 for f in r['top_features'] if f['description'])
    print(f"Example {i+1}:")
    print(f"  - L0: {r['l0']}")
    print(f"  - Features with descriptions: {features_with_desc}/{len(r['top_features'])}")
    print(f"  - Correct answer: {r['correct_answer_text']}")